# Malaria Detection: Try It Yourself
Loads the already-trained model, so there is **no training** here. Just run the cells top to bottom and upload cell images.

CPU runtime is fine. No GPU needed.

## 1. Load the model

In [ ]:
import os, io
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from PIL import Image

IMG_SIZE = 128
MODEL_PATH = "/content/drive/MyDrive/malaria_project/malaria_model.keras"

**Option A (Drive):** run the next cell if the model is in Google Drive under `malaria_project/`.

**Option B (upload):** skip it and run the cell after instead, then pick `malaria_model.keras` from your computer.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Option B: only runs if the model wasn't found in Drive
if not os.path.exists(MODEL_PATH):
    from google.colab import files
    print("Model not found in Drive. Upload malaria_model.keras:")
    up = files.upload()
    MODEL_PATH = list(up.keys())[0]

model = tf.keras.models.load_model(MODEL_PATH, compile=False)
print("Model loaded")

## 2. Test images
Click **Choose files** and select one or more cell images (png / jpg). Works best on single-cell crops like the ones in the dataset, not whole microscope slides.

In [ ]:
from google.colab import files

def predict_pil(img):
    img = img.convert("RGB").resize((IMG_SIZE, IMG_SIZE))
    arr = np.array(img, dtype="float32")[None]
    return float(model.predict(arr, verbose=0)[0][0])

uploaded = files.upload()
n = len(uploaded)
fig, ax = plt.subplots(1, n, figsize=(3.5 * n, 4), squeeze=False)
for a, (name, data) in zip(ax[0], uploaded.items()):
    img = Image.open(io.BytesIO(data))
    prob = predict_pil(img)
    infected = prob >= 0.5
    a.imshow(img)
    a.set_title(f"{'Parasitized' if infected else 'Uninfected'}\n{(prob if infected else 1 - prob):.1%} confident",
                color="red" if infected else "green")
    a.axis("off")
plt.tight_layout(); plt.show()

## 3. Optional: web app
Gives a link with drag-and-drop upload, which is handy for a live demo. Run both cells, then open the **public URL** printed at the bottom.

In [ ]:
!pip install -q gradio

In [ ]:
import gradio as gr

def classify(img):
    prob = predict_pil(img)
    return {"Parasitized": prob, "Uninfected": 1 - prob}

gr.Interface(
    fn=classify,
    inputs=gr.Image(type="pil", label="Upload a cell image"),
    outputs=gr.Label(num_top_classes=2, label="Prediction"),
    title="Malaria Cell Detector",
    description="Upload a single blood smear cell image to check if it is parasitized.",
).launch(share=True)